# מחברת 01 — להבין את הדאטה לפני שבונים פיצ׳רים

המטרה: להכיר את התצוגות, הכיסוי והקשרים בין משחק, מחזור ועונה. אין כאן עיבוד פיצ׳רים, אימון או בחירת מודל.

**דאטה מאומת → מחברת 01: היכרות → מחברת 02: פיצ׳רים → מחברת 03: אימון והערכה.**

## איך לקרוא את המחברת

ארבע קבוצות עמודות: **identity** — מי ומתי; **pre_match** — מידע ידוע לפני השריקה; **in_match** — מה שקרה במשחק; **data_quality** — מגבלות המקור.

שלושה תפקידי עונות: **development** (2006/07–2023/24) למחקר ולכיוונון; **final_validation** (2024/25) לבדיקה אחרונה אחת; **test** (2025/26) למבחן סופי אחד. העונות האחרונות מוצגות בהקשר ברור, ולא מצטרפות לסטטיסטיקות המחקר.

2026/27 בחוץ כי זו עונה חיה ולא שלמה. 01 עוסקת בהבנה; 02 תיצור פיצ׳רים ממשחק, ממחזור ומעונה; 03 תאמן לפי תוכנית הוולידציה.

## הגדרות
הדוגמאות יבואו מעונת development השלמה 2023/24. הצבעים קבועים לתפקידי העונות ולתוצאות משחק.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import yaml
from IPython.display import display

from ligat.analysis.prepare import (
    ALL_SEASONS,
    COLUMN_GROUPS,
    DEVELOPMENT_SEASONS,
    check_analysis_tables,
    data_availability,
    describe_missing_values,
    load_analysis_tables,
    outcome_base_rates,
    team_history,
    team_profile,
    teams_overview,
)
from ligat.data.repository import Repository
from ligat.evaluate.splits import expanding_window_folds

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", None)
pio.renderers.default = "plotly_mimetype+notebook"
EXAMPLE_SEASON = 25
TEAM = "מכבי חיפה"
role_colors = {"development": "#2563eb", "final_validation": "#7c3aed", "test": "#f59e0b"}
outcome_colors = {"בית": "#2563eb", "תיקו": "#6b7280", "חוץ": "#f59e0b"}
print("טעינה: 2006/07–2025/26; מחקר: development בלבד; דוגמאות: 2023/24.")

## טעינת התצוגות
כל הנתונים נקראים דרך Repository עם סינון. אין שינוי בנתונים הנקיים. תצוגות המאמנים נשארות בחבילה, אך אינן נושא במחברת.

In [ ]:
repository = Repository()
tables = load_analysis_tables(repository)
visible_tables = {
    "match_table": "משחק",
    "team_matches": "קבוצה במשחק",
    "goal_events": "שער",
    "team_seasons": "קבוצה בעונה",
    "season_summary": "עונה",
}
display(
    pd.DataFrame(
        [
            {
                "טבלה": name,
                "שורה =": unit,
                "מפתח": ", ".join(getattr(tables, name).index.names),
                "שורות": len(getattr(tables, name)),
                "עמודות": len(getattr(tables, name).columns),
            }
            for name, unit in visible_tables.items()
        ]
    )
)

## ציר הזמן של הדאטה
מספרי המשחקים מציגים כיסוי, לא ביצועי מודל. העונות האחרונות מסומנות לפי תפקידן כדי שלא יתערבבו במחקר.

In [ ]:
timeline = (
    tables.match_table.groupby(["season_id", "season_role"], observed=True)
    .agg(n_matches=("date", "size"), first_date=("date", "min"), last_date=("date", "max"))
    .reset_index()
)
timeline["עונה"] = timeline.season_id.map(lambda s: f"{s + 1998}/{str(s + 1999)[-2:]}")
display(timeline[["עונה", "season_role", "n_matches"]])

השאלה: אילו עונות שלמות זמינות ומה תפקיד כל אחת? הטווחים מציגים את מועד המשחק הראשון והאחרון שנשמרו.

In [ ]:
fig = px.timeline(
    timeline,
    x_start="first_date",
    x_end="last_date",
    y="עונה",
    color="season_role",
    color_discrete_map=role_colors,
    hover_data=["n_matches"],
    title="18 עונות לפיתוח, עונה לוולידציה אחרונה ועונה למבחן סופי",
)
fig.update_layout(
    template="plotly_white", xaxis_title="תאריך", legend_title="תפקיד העונה", height=620
)
fig.update_yaxes(autorange="reversed")
fig.show()

## תוכנית הוולידציה
חלון האימון מתחיל בשלוש עונות ומתרחב; בכל fold בודקים את העונה הבאה. בוחרים היפר־פרמטרים פעם אחת לפי ממוצע ההפסד בכל 15 ה־folds, משוקלל במספר המשחקים. שתי הבדיקות האחרונות אינן משמשות לכיוונון.

In [ ]:
folds = expanding_window_folds(repository=repository)
display(folds)

השאלה: האם כל אימון קודם לבדיקה שלו? כל שורה מציגה חלון אימון ומיד אחריו עונת בדיקה, ללא ערבוב אקראי.

In [ ]:
fold_labels = [f"fold {v}" if isinstance(v, int) else str(v) for v in folds.fold]
grid = []
for fold in folds.itertuples():
    validation_color = 3 if fold.fold == "final_validation" else 4 if fold.fold == "test" else 2
    grid.append(
        [
            1 if s in fold.train_seasons else validation_color if s == fold.validation_season else 0
            for s in ALL_SEASONS
        ]
    )
season_labels = [f"{s + 1998}/{str(s + 1999)[-2:]}" for s in ALL_SEASONS]
fig = go.Figure(
    go.Heatmap(
        z=grid,
        x=season_labels,
        y=fold_labels,
        zmin=0,
        zmax=4,
        colorscale=[
            [0, "white"],
            [0.124, "white"],
            [0.125, "#2563eb"],
            [0.374, "#2563eb"],
            [0.375, "#fb923c"],
            [0.624, "#fb923c"],
            [0.625, "#7c3aed"],
            [0.874, "#7c3aed"],
            [0.875, "#f59e0b"],
            [1, "#f59e0b"],
        ],
        colorbar=dict(
            tickvals=[0, 1, 2, 3, 4],
            ticktext=["לא בשימוש", "אימון", "ולידציה בפיתוח", "ולידציה אחרונה", "מבחן סופי"],
        ),
    )
)
fig.update_layout(
    template="plotly_white",
    title="האימון מתרחב; כל עונת בדיקה נמצאת אחריו",
    xaxis_title="עונה",
    yaxis_title="שלב ההערכה",
    height=600,
)
fig.update_yaxes(autorange="reversed")
fig.show()

## משחקים: דוגמה מ־2023/24
נראה תוצאה ומחצית ליד הקבוצות והתאריך. כל NA משקף נתון חסר או חריג מקור, ולא אפס מומצא.

In [ ]:
example_matches = tables.match_table.loc[tables.match_table.season_id.eq(EXAMPLE_SEASON)]
match_columns = [
    "date",
    "home_team_name",
    "away_team_name",
    "home_goals",
    "away_goals",
    "ht_home_goals",
    "ht_away_goals",
    "home_red_cards",
    "away_red_cards",
]
display(example_matches[match_columns].head(8))

## קבוצות העמודות
pre_match ידוע מראש; in_match הוא יעד או חומר להיסטוריה של משחקים הבאים. מחברת 02 תעבד פיצ׳רים — כאן רק מכירים את העמודות הקיימות.

In [ ]:
display(
    pd.DataFrame(
        [{"קבוצה": name, "מספר עמודות": len(columns)} for name, columns in COLUMN_GROUPS.items()]
    )
)

נציג נתונים ידועים מראש. NA בשווי t−1 יכול לנבוע מעלייה לליגה; NA בשופט הוא חוסר במקור. אין השלמה במחברת הזו.

In [ ]:
pre_columns = [c for c in COLUMN_GROUPS["pre_match"] if "coach" not in c]
display(example_matches[pre_columns].head(5))

## שתי נקודות מבט לכל משחק
לכל קבוצה יש שורה משלה. NA בשער ראשון מופיע ב־0:0 או כשסדר השערים אינו ברור; NA במחצית מציין סתירה מתועדת.

In [ ]:
example_teams = tables.team_matches.loc[tables.team_matches.season_id.eq(EXAMPLE_SEASON)]
team_columns = [
    "date",
    "team_name",
    "opponent_name",
    "is_home",
    "goals_scored",
    "goals_conceded",
    "points_earned",
    "ht_state",
]
display(example_teams[team_columns].head(8))

אותו משחק מופיע פעם מנקודת הבית ופעם מנקודת החוץ. כך אפשר להכיר את מבנה הקלט לקבוצה ללא שני מסלולי קוד נפרדים.

In [ ]:
example_game = example_matches.index[0]
display(example_teams.xs(example_game, level="game_id")[team_columns])

## שערים כאירועים
NA ב־half או במצב שלפני השער מסמן סתירת מחצית או שערים באותה דקה. added_time אינה מוצגת, כי היא תמיד חסרה במקור; דקות רצות מוסברות באמצעות half.

In [ ]:
example_goals = tables.goal_events.loc[tables.goal_events.season_id.eq(EXAMPLE_SEASON)]
display(example_goals.drop(columns="added_time").head(10))

השאלה: באיזה חלק במשחק מובקעים השערים בתקופת הפיתוח? אין בגרף שערים מעונות הוולידציה והמבחן; טווח לא ידוע אינו מושלם.

In [ ]:
development_goals = tables.goal_events.loc[tables.goal_events.season_id.isin(DEVELOPMENT_SEASONS)]
minute_counts = (
    development_goals.minute_bucket.value_counts(sort=False).rename("שערים").reset_index()
)
minute_counts["טווח דקות"] = minute_counts.minute_bucket.astype("string").replace(
    {"45+ (first-half stoppage)": "45+ (תוספת מחצית)"}
)
largest_bucket = minute_counts.loc[minute_counts["שערים"].idxmax(), "טווח דקות"]
fig = px.bar(
    minute_counts,
    x="טווח דקות",
    y="שערים",
    color_discrete_sequence=["#6b7280"],
    title=f"הטווח עם הכי הרבה שערים בפיתוח: {largest_bucket}",
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero")
fig.show()

## קבוצה בעונה
דוגמה מעונת 2023/24, לפי מיקום סופי. NA בנתח שווי קודם מסמן עולה או שווי חסר; נתח נוכחי הוא תיאור העונה בלבד.

In [ ]:
example_seasons = tables.team_seasons.xs(EXAMPLE_SEASON, level="season_id").sort_values(
    "final_position"
)
display(
    example_seasons[
        [
            "team_name",
            "final_position",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "value_share_of_league",
            "prev_season_value_share",
        ]
    ]
    .round(3)
    .head(10)
)

## סיכום עונות הפיתוח
נציג רק עמודות מרכזיות ובדיוק קריא. נקודות האלופה וקו הירידה מחושבות על המגרש, ללא קיזוז והורדות מנהלתיות.

In [ ]:
development_summary = tables.season_summary.loc[tables.season_summary.season_role.eq("development")]
summary_columns = [
    "n_teams",
    "n_matches",
    "goals_per_match",
    "home_win_rate",
    "draw_rate",
    "away_win_rate",
    "champion_points_per_match",
    "relegation_line_points_per_match",
]
display(development_summary[summary_columns].round(3))

השאלה: האם שיעורי התוצאות אחידים בין עונות הפיתוח? זהו תיאור מחקרי של development בלבד, בלי שימוש בתוצאות העונות הסופיות.

In [ ]:
rates = development_summary[["home_win_rate", "draw_rate", "away_win_rate"]].rename(
    columns={"home_win_rate": "בית", "draw_rate": "תיקו", "away_win_rate": "חוץ"}
)
rates.index = [f"{s + 1998}/{str(s + 1999)[-2:]}" for s in rates.index]
rates = (
    rates.mul(100)
    .rename_axis("עונה")
    .reset_index()
    .melt(id_vars="עונה", var_name="תוצאה", value_name="% מהמשחקים")
)
low = development_summary.home_win_rate.min()
high = development_summary.home_win_rate.max()
fig = px.bar(
    rates,
    x="עונה",
    y="% מהמשחקים",
    color="תוצאה",
    color_discrete_map=outcome_colors,
    title=f"בפיתוח שיעור ניצחונות הבית נע בין {low:.0%} ל־{high:.0%}",
)
fig.update_layout(template="plotly_white", barmode="stack", yaxis_range=[0, 100])
fig.show()

## מפת זמינות
הזמינות היא אחוז משחקים עם מידע תקין; למאמנים ולשווי קודם מחשבים ממוצע של שני הצדדים. NA אינו מושלם. מאמנים הם נתון בעדיפות נמוכה; ההחלטה תתקבל ב־ablation — השוואה עם ובלי הפיצ׳ר.

In [ ]:
availability = data_availability(tables)
display(availability.round(1))

השאלה: אילו שדות זמינים בכל עונה? הכיסוי ההיסטורי של הרכבים ומאמנים חלקי; שווי קודם חסר בתחילת התקופה ולעולות. מסגרות סגולה וזהובה מסמנות ולידציה אחרונה ומבחן סופי.

In [ ]:
fields = {
    "halftime": "מחצית תקינה",
    "scorers": "מבקיעים",
    "timeline": "ציר אירועים",
    "lineups": "הרכבים",
    "referee": "שופט",
    "coach": "מאמן",
    "prev_season_value": "שווי עונה קודמת",
}
coverage_matrix = availability[list(fields)].rename(columns=fields).T
coverage_matrix.columns = season_labels
fig = px.imshow(
    coverage_matrix,
    color_continuous_scale="Blues",
    zmin=0,
    zmax=100,
    labels={"x": "עונה", "y": "שדה", "color": "% זמין"},
    aspect="auto",
    title="הכיסוי משתפר עם השנים; מאמנים ושווי קודם נשארים חלקיים",
)
for season, color in [(26, role_colors["final_validation"]), (27, role_colors["test"])]:
    position = list(ALL_SEASONS).index(season)
    fig.add_vrect(
        x0=position - 0.5,
        x1=position + 0.5,
        line_width=3,
        line_color=color,
        fillcolor="rgba(0,0,0,0)",
    )
fig.update_layout(template="plotly_white")
fig.show()

מתחת למפה נציג רשימת חוסרים מקוצרת עם הסברים. NA בהסבר של עמודה ללא חוסר אינו בעיה; כאן מוצגות רק שורות שדורשות זהירות, בלי שדות מאמנים ובלי added_time הריקה תמיד.

In [ ]:
missing = describe_missing_values(tables)
visible_missing = missing.loc[
    missing.status.ne("ok")
    & ~missing.column.str.contains("coach|added_time")
    & missing["table"].ne("coach_spells")
]
display(visible_missing[["table", "column", "n_missing", "share", "reason"]].head(12))

## כל הקבוצות — development בלבד
הטבלה מסכמת את כל הקבוצות בתקופת הפיתוח, לפי נקודות למשחק. ממוצעי תוצאות משוקללים במשחקים; NA בממוצע שווי או בבית/חוץ משקף מידע חסר, לא אפס.

In [ ]:
overview = teams_overview(tables)
display(overview.round(3))

השאלה: באילו עונות כל קבוצה הייתה בליגה ומה היו הנקודות על המגרש? תא ריק אומר שהקבוצה לא בליגה. העונות הסופיות ממוסגרות כמידע הקשר בלבד, ואינן משתתפות בסיכום כל הקבוצות.

In [ ]:
team_matrix = (
    tables.team_seasons.reset_index()
    .pivot(index="team_name", columns="season_id", values="points_per_match")
    .reindex(columns=list(ALL_SEASONS))
)
team_matrix.columns = season_labels
fig = px.imshow(
    team_matrix,
    color_continuous_scale="Blues",
    aspect="auto",
    labels={"x": "עונה", "y": "קבוצה", "color": "נקודות למשחק"},
    title="תאים ריקים מסמנים עונות שבהן הקבוצה לא שיחקה בליגת העל",
)
for season, color in [(26, role_colors["final_validation"]), (27, role_colors["test"])]:
    position = list(ALL_SEASONS).index(season)
    fig.add_vrect(
        x0=position - 0.5,
        x1=position + 0.5,
        line_width=3,
        line_color=color,
        fillcolor="rgba(0,0,0,0)",
    )
fig.update_layout(template="plotly_white", height=900)
fig.show()

## פרופיל קבוצה
שנו את TEAM בתא ההגדרות כדי לראות קבוצה אחרת. הפרופיל כולל עונות סופיות בהקשר מסומן; NA ב־promoted_in בעונה הראשונה אומר שאין עונה קודמת להשוואה.

In [ ]:
profile = team_profile(tables, TEAM)
display(profile.round(3))

השאלה: איך השתנה המיקום הסופי של הקבוצה? מקום 1 מוצג למעלה. קו אפור מחבר את ההיסטוריה, והנקודות צבועות לפי תפקיד העונה.

In [ ]:
profile_plot = profile.reset_index()
profile_plot["עונה"] = profile_plot.season_id.map(lambda s: f"{s + 1998}/{str(s + 1999)[-2:]}")
fig = px.scatter(
    profile_plot,
    x="עונה",
    y="final_position",
    color="season_role",
    color_discrete_map=role_colors,
    title=f"{TEAM}: היסטוריית המיקום הסופי",
    labels={"final_position": "מיקום סופי", "season_role": "תפקיד העונה"},
)
fig.add_trace(
    go.Scatter(
        x=profile_plot["עונה"],
        y=profile_plot.final_position,
        mode="lines",
        line=dict(color="#9ca3af"),
        showlegend=False,
    )
)
fig.update_layout(template="plotly_white", yaxis_autorange="reversed")
fig.show()

השאלה: כמה נקודות למשחק צברה הקבוצה בכל עונה? אלה נקודות ממשחקים שנשמרו, ולא הנקודות הרשמיות לאחר עונשים.

In [ ]:
fig = px.bar(
    profile_plot,
    x="עונה",
    y="points_per_match",
    color="season_role",
    color_discrete_map=role_colors,
    title=f"{TEAM}: נקודות על המגרש למשחק בכל עונה",
    labels={"points_per_match": "נקודות למשחק", "season_role": "תפקיד העונה"},
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero")
fig.show()

נחזור לפרופיל הקבוצה בתוך 2023/24. נקודות מצטברות כאן הן תיאור של משחקים, לא פיצ׳ר. אין בפלט הזה עונות סופיות.

In [ ]:
history = team_history(tables.team_matches, TEAM)
team_example = history.loc[history.season_id.eq(EXAMPLE_SEASON)].copy()
display(team_example[team_columns].tail(10))

השאלה: איך הצטברו הנקודות של הקבוצה בעונת הדוגמה? מחברת 02 תחליט אילו סיכומי עבר הופכים לפיצ׳רים.

In [ ]:
team_example["נקודות מצטברות"] = team_example.points_earned.cumsum()
fig = px.line(
    team_example,
    x="date",
    y="נקודות מצטברות",
    color_discrete_sequence=["#2563eb"],
    title=f"{TEAM}: צבירת נקודות על המגרש ב־2023/24",
    labels={"date": "תאריך"},
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero")
fig.show()

## מידע ברמת מחזור ועונה
הטבלה הרשמית אחרי מחזור 13 ב־2023/24 מתארת את מצב הליגה באותו שלב. השמות הם לתצוגה; מזהי הקבוצות הם המפתחות. אנחנו מציגים את הטבלה, בלי לבנות ממנה פיצ׳רים.

In [ ]:
round_table = repository.read_filtered(
    "round_tables", [("season_id", "==", EXAMPLE_SEASON), ("round", "==", 13)]
)
names = tables.team_seasons.reset_index().drop_duplicates("team_id").set_index("team_id").team_name
round_table["team_name"] = round_table.team_id.map(names)
display(
    round_table[
        [
            "position",
            "team_id",
            "team_name",
            "played",
            "won",
            "drawn",
            "lost",
            "points_official",
            "points_model",
        ]
    ].sort_values("position")
)

ההקשר העונתי כולל מיקום קודם, עלייה או ירידה ופורמט הליגה. NA במיקום קודם מציין קבוצה שלא הייתה בליגה בעונה הקודמת. מחברת 02 תיצור מיקום לפני המשחק, פערים מהמקום הראשון/השישי/הירידה, מחזורים שנותרו, שלב עונה, מיקום קודם, עלייה ונתח שווי — לא כאן.

In [ ]:
context = tables.team_seasons.xs(EXAMPLE_SEASON, level="season_id")[
    ["team_name", "final_position", "final_stage", "promoted_in", "relegated"]
].copy()
previous_positions = tables.team_seasons.xs(EXAMPLE_SEASON - 1, level="season_id").final_position
context["previous_final_position"] = previous_positions.reindex(context.index).astype("Int64")
context["n_teams"] = int(tables.season_summary.loc[EXAMPLE_SEASON, "n_teams"])
season_format = next(
    s
    for s in yaml.safe_load(Path("config/seasons.yaml").read_text(encoding="utf-8"))["seasons"]
    if s["season_id"] == EXAMPLE_SEASON
)
context["playoff_split_after_round"] = season_format["regular_rounds"]
display(context.sort_values("final_position"))

## בדיקות בריאות
הבדיקות מאמתות מבנה, לוח משחקים, שערים והצלבות בכל העונות השלמות. חוסרים מתועדים נשארים אזהרות; בדיקת תקינות אינה שימוש בתוצאות לכיוונון.

In [ ]:
warnings = check_analysis_tables(tables)
display(
    pd.DataFrame(
        {
            "בדיקה": [
                "מפתחות והיקף",
                "משחקים ונקודות",
                "לוח משחקים והחרגות",
                "שערים ומחצית",
                "כיסוי ומקורות",
            ],
            "מצב": "עבר",
        }
    )
)
display(pd.DataFrame({"אזהרות מתועדות": [len(warnings)]}))

## בנצ׳מרק B0 — development בלבד
שיעורי הבסיס הם תיאור של תקופת הפיתוח. רווחי Wilson של 95% נפרדים לכל תוצאה; בבקטסט נדרש לחשב את השיעורים מאימון בלבד.

In [ ]:
development_matches = tables.match_table.loc[tables.match_table.season_role.eq("development")]
base_rates = outcome_base_rates(development_matches)
overall = base_rates.loc[base_rates.scope.eq("overall")].drop(columns=["season_id", "scope"])
display(overall.round(4))

נציג בנפרד את שיעורי B0 לפי עונה, בלי עמודות חסרות. עונות 26–27 אינן נכללות בטבלה.

In [ ]:
display(
    base_rates.loc[base_rates.scope.eq("season")]
    .pivot(index="season_id", columns="outcome", values="rate")
    .round(3)
)

השאלה: איזו תוצאה שכיחה בפיתוח ומה אי־הוודאות בשיעור שלה? אותם צבעים מציינים בית/תיקו/חוץ בכל המחברת.

In [ ]:
b0_plot = overall.copy()
b0_plot["תוצאה"] = b0_plot.outcome.map({"home_win": "בית", "draw": "תיקו", "away_win": "חוץ"})
b0_plot["% מהמשחקים"] = b0_plot.rate * 100
b0_plot["עליונה"] = (b0_plot.ci_upper - b0_plot.rate).clip(lower=0) * 100
b0_plot["תחתונה"] = (b0_plot.rate - b0_plot.ci_lower).clip(lower=0) * 100
winner = b0_plot.loc[b0_plot.rate.idxmax()]
fig = px.bar(
    b0_plot,
    x="תוצאה",
    y="% מהמשחקים",
    color="תוצאה",
    color_discrete_map=outcome_colors,
    error_y="עליונה",
    error_y_minus="תחתונה",
    title=f"בפיתוח {winner['תוצאה']} היא התוצאה השכיחה: {winner.rate:.1%}",
)
fig.update_layout(template="plotly_white", yaxis_rangemode="tozero")
fig.show()

## מה למדנו ומה מגיע עכשיו?

- season_role מפריד בין נתונים שאפשר להכיר לבין נתונים המשמשים למחקר ולכיוונון.
- יש מידע בשלוש רמות: משחק, מצב הליגה אחרי מחזור והקשר עונתי.
- תאי זמינות ריקים אינם אפס; כיסוי מוקדם, עולות וחריגי מקור מסבירים את החוסרים.
- חלון האימון מתרחב, ובחירת ההיפר־פרמטרים נשענת על ממוצע משוקלל של 15 folds.

**02:** עיבוד פיצ׳רים ממשחק, ממחזור ומעונה, עם מניעת דליפה. **03:** אימון והערכה לפי התוכנית, ולידציה אחרונה אחת ומבחן סופי אחד.

הכרעות על פיצ׳רים, מודלים או השלמות יתקבלו במשימות הבאות; אין כאלה במחברת 01.